output_attentions

In [ ]:
## 생성된 토큰을 기준으로 attention 값을 측정
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
        output_attentions=True
    )

text = "The weather is cloudy today, so it feels like cold"
inputs = tokenizer(text,return_tensors="pt").to("cuda")

with torch.no_grads():
  outputs = model(**inputs)

attentions = outputs.attentions

print(f"Layer 개수: {len(attentions)}")
print(f"Attention Shape: {attentions[0].shape}")

Transformer

In [6]:
import torch

d_model = 512

def MHA(h, Q, K, V, Wq, Wk, Wv ,Wo):
  # Wk : d_model * d_model
  # Wk_i : d_model * dk
  # d_k: d_model / h
  # 논문에서는 dk = dv = d_model/h로 설정함.
  heads = []
  d_k = d_model // h

  for i in range(h):
    Wq_i = Wq[:, i*d_k:(i+1)*d_k]
    Wk_i = Wk[:, i*d_k:(i+1)*d_k]
    Wv_i = Wv[:, i*d_k:(i+1)*d_k]

    heads.append(attn(torch.matmul(Q, Wq_i), torch.matmul(K, Wk_i), torch.matmul(V, Wv_i)))

  # torch.cat(dim=-1) : (seq_len, d_model)
  return torch.matmul(torch.cat(heads, dim=-1), Wo)

def attn(Q, K, V):
  d_k = Q.shape[-1]
  return torch.matmul(torch.softmax(torch.matmul(Q, K.transpose(-2, -1)) / d_k ** (1/2), dim=-1), V)

def MMHA(h, Q, K, V, Wq, Wk, Wv ,Wo):
  heads = []
  d_k = d_model // h

  for i in range(h):
    Wq_i = Wq[:, i*d_k:(i+1)*d_k]
    Wk_i = Wk[:, i*d_k:(i+1)*d_k]
    Wv_i = Wv[:, i*d_k:(i+1)*d_k]

    heads.append(masked_attn(torch.matmul(Q, Wq_i), torch.matmul(K, Wk_i), torch.matmul(V, Wv_i)))

def masked_attn(Q, K, V):
  d_k = Q.shape[-1]

  qk_T = torch.matmul(Q, K.transpose(-2, -1)) / d_k ** (1/2)
  # masking
  masked = torch.triu(torch.ones_like(qk_T), diagonal=1)
  masked_qk_T = qk_T.masked_fill(masked == 1, float('-inf'))
  attn_score = torch.softmax(masked_qk_T, dim=-1)
  return torch.matmul(attn_score, V)

In [5]:
import torch.nn as nn
class Encoder(nn.Module):
  def __init__(self):
    super().__init__()
    self.Wq = nn.Parameter(torch.randn(d_model, d_model))
    self.Wk = nn.Parameter(torch.randn(d_model, d_model))
    self.Wv = nn.Parameter(torch.randn(d_model, d_model))
    self.Wo = nn.Parameter(torch.randn(d_model, d_model))

    self.linear1 = nn.Linear(d_model, 4*d_model)
    self.linear2 = nn.Linear(4*d_model, d_model)

    self.norm1 = nn.LayerNorm(d_model)
    self.norm2 = nn.LayerNorm(d_model)

  def forward(self, x):
    x_mid = self.norm1(x + MHA(8, x, x, x, self.Wq, self.Wk, self.Wv, self.Wo))
    ffn = self.linear2(torch.relu(self.linear1(x_mid))) # torch.matmul(torch.relu(torch.matmul(x, W1) + b1), W2) + b2
    x_out = self.norm2(x_mid + ffn)
    return x_out

In [ ]:
class Decoder(nn.Module):
  def __init__(self, enc_out):
    super().__init__()
    self.Wq1 = nn.Parameter(torch.randn(d_model, d_model))
    self.Wk1 = nn.Parameter(torch.randn(d_model, d_model))
    self.Wv1 = nn.Parameter(torch.randn(d_model, d_model))
    self.Wo1 = nn.Parameter(torch.randn(d_model, d_model))

    self.Wq2 = nn.Parameter(torch.randn(d_model, d_model))
    self.Wk2 = nn.Parameter(torch.randn(d_model, d_model))
    self.Wv2 = nn.Parameter(torch.randn(d_model, d_model))
    self.Wo2 = nn.Parameter(torch.randn(d_model, d_model))

    self.enc_out = enc_out

    self.linear1 = nn.Linear(d_model, 4*d_model)
    self.linear2 = nn.Linear(4*d_model, d_model)

    self.norm1 = nn.LayerNorm(d_model)
    self.norm2 = nn.LayerNorm(d_model)
    self.norm3 = nn.LayerNorm(d_model)

  def forward(self, x):
    x_1 = self.norm1(x + MMHA(8, x, x, x, self.Wq1, self.Wk1, self.Wv1, self.Wo1))
    x_2 = self.norm2(x_1 + MHA(8, x_1, self.enc_out, self.enc_out, self.Wq2, self.Wk2, self.Wv2, self.Wo2))
    ffn = self.linear2(torch.relu(self.linear1(x_2)))
    x_out = self.norm3(x_2 + ffn)
    return x_out
